In [2]:
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

import numpy as np
import pandas as pd
import time
import os
import torch
import torch.nn as nn
import io
from sklearn.metrics import average_precision_score


In [ ]:
from fsd50k_setup import (
    train_loader, val_loader, eval_loader,
    NUM_CLASSES, decode_labels, encode_labels,
    FRAME_COUNT, vocab, top_level, get_top_level_family, train_df, focalloss, prior_bias, mixup
)

# Sanity check
features, labels = next(iter(train_loader))
print(f"Feature batch shape : {features.shape}")
print(f"Label batch shape   : {labels.shape}")
print(f"First clip labels   : {decode_labels(labels[0])}")

FileNotFoundError: [Errno 2] No such file or directory: 'preprocessed\\train\\162755.pt'

### Build model

In [ ]:
class AudioCNN(nn.Module):
    """
    Four convolutional blocks followed by global average pooling
    and a linear classifier.

    Each block:
        Conv2d → BatchNorm → ReLU → MaxPool """


    def __init__(self, num_classes: int = 200): #dim comments assumes N_mels = 128
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 — learns low-level features (edges in freq/time)
            # Input:  (batch, 1,   128, 516)
            # Output: (batch, 32,  64,  258)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2 — learns mid-level patterns (harmonics, rhythms)
            # Input:  (batch, 32,  64,  258)
            # Output: (batch, 64,  32,  129)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3 — learns higher-level combinations
            # Input:  (batch, 64,  32,  129)
            # Output: (batch, 128, 16, 64)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4 — learns abstract sound representations
            # Input:  (batch, 128, 16,  64)
            # Output: (batch, 128, 8,   32)
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((4, 4))  #(batch, 128, 8, 32) → (batch, 128, 4, 4)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 512), nn.ReLU(),
                                      nn.Dropout(0.3), nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
                                      nn.Linear(256, num_classes))

    def forward(self, x):
      x = self.features(x)    # (batch, 128, 8, 32)
      x = self.gap(x)         # (batch, 128, 4, 4)
      x = x.flatten(1)        # (batch, 2048)
      x = self.classifier(x)  # (batch, 200)
      return x

### Training and evaluation setup

In [ ]:
# ─────────────────────────────────────────────
# TRAINING SETUP
# ─────────────────────────────────────────────
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Using device: {device}")

NUM_EPOCHS = 100
model     = AudioCNN(num_classes=NUM_CLASSES).to(device)

# Quick parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# Compute pos_weight from training labels
all_labels = torch.stack([encode_labels(lbl) for lbl in train_df["labels"]])
pos        = all_labels.sum(dim=0).clamp(min=1)
neg        = len(all_labels) - pos
pos_weight = (neg / pos).to(device)   # shape: (num_classes,)
pos_weight = (neg / pos).clamp(max=10.0).to(device)

# Focal loss with pos_weight to handle class imbalance
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma      = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce  = nn.functional.binary_cross_entropy_with_logits(
                   logits, targets, pos_weight=self.pos_weight, reduction='none')
        prob = torch.sigmoid(logits)
        p_t  = targets * prob + (1 - targets) * (1 - prob)
        return (bce * (1 - p_t) ** self.gamma).mean()

if focalloss:
    criterion = FocalLoss(gamma=2.0, pos_weight=pos_weight)
else:
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

#introduce prior bias to help with class imbalance
if prior_bias:
    prior = all_labels.mean(dim=0).clamp(1e-4, 1 - 1e-4)
    init_bias  = torch.log(prior / (1 - prior))   # inverse sigmoid of prior
    model.classifier[-1].bias.data = init_bias.to(device)

print(f"pos_weight min:  {pos_weight.min().item():.1f}")
print(f"pos_weight max:  {pos_weight.max().item():.1f}")
print(f"pos_weight mean: {pos_weight.mean().item():.1f}")


# ─────────────────────────────────────────────
# TRAINING AND VALIDATION FUNCTIONS
# ─────────────────────────────────────────────
def train_one_epoch(loader, alpha=0.4):
    model.train()
    total_loss = 0.0

    for batch_idx, (features, labels) in enumerate(loader):
        features = features.to(device, non_blocking=True)
        labels   = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)

        if mixup:
            lam      = np.random.beta(alpha, alpha)
            idx      = torch.randperm(features.size(0)).to(device)
            features = lam * features + (1 - lam) * features[idx]
            labels   = lam * labels   + (1 - lam) * labels[idx]

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device, non_blocking=True)
            labels   = labels.to(device, non_blocking=True)
            logits   = model(features)
            total_loss += criterion(logits, labels).item()
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.vstack(all_probs)   # (N, 200)
    all_labels = np.vstack(all_labels)  # (N, 200)

    # mAP — the standard FSD50K metric
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UndefinedMetricWarning)
        mAP = average_precision_score(all_labels, all_probs, average="macro")


    return total_loss / len(loader), mAP

Using device: cpu
Trainable parameters: 266,760
pos_weight min:  2.2
pos_weight max:  10.0
pos_weight mean: 9.9


### Rough estimate of runtime

In [ ]:
# # Check 1 — are we actually on CPU?
# print(f"Device: {device}")

# # Check 2 — how much RAM is being used?
# import psutil
# ram = psutil.virtual_memory()
# print(f"RAM used: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB")
# print(f"RAM available: {ram.available/1e9:.1f} GB")

# # Check 3 — how loaded is the CPU during a forward pass?
# start = time.time()
# for i, (features, labels) in enumerate(train_loader):
#     features = features.to(device)
#     labels = labels.to(device)

#     optimizer.zero_grad()
#     logits = model(features)
#     loss = criterion(logits, labels)
#     loss.backward()
#     optimizer.step()

#     if i == 16:
#         break

# elapsed = time.time() - start
# print(f"16 training batches: {elapsed:.2f}s")
# print(f"Estimated epoch: {elapsed * len(train_loader)/16/60:.1f} min")

# #Then reset model and optimizer to start training for real
# model = AudioCNN(num_classes=NUM_CLASSES).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Run training loop

In [ ]:
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
# ─────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────
model_name  = "full_model_1"
description = "100 epochs, no oversampling, no spec augment, random crop, with scheduler, with focal loss, with bias initialization, early stopping with patience 10, first 6 seconds, n_mels=64"
log_file    = "training_log.csv"

if os.path.exists(f"{model_name}.pt"): #make sure to remove old model file if it exists, otherwise we might accidentally load it later and get confused
    os.remove(f"{model_name}.pt")

best_mAP         = 0.0
patience         = 15
patience_counter = 0
# NUM_EPOCHS       = 20 ---- this is defined above for the scheduler
history          = []

print(f"\nTraining for {NUM_EPOCHS} epochs...\n")
print(f"{'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14} {'Val mAP'}")
print("-" * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss        = train_one_epoch(train_loader)
    val_loss, val_mAP = evaluate(val_loader)
    scheduler.step() # update learning rate

    history.append({
        "epoch":      epoch,
        "train_loss": round(train_loss, 4),
        "val_loss":   round(val_loss,   4),
        "val_mAP":    round(val_mAP,    4),
    })

    if val_mAP > best_mAP:
        best_mAP = val_mAP
        patience_counter = 0
        torch.save(model.state_dict(), f"/content/drive/MyDrive/ML_Final_Project/{model_name}.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"{epoch:<8} {train_loss:<14.4f} {val_loss:<14.4f} {val_mAP:.4f}")

print(f"\nBest val mAP: {best_mAP:.4f}")

# Save to training log
df_history = pd.DataFrame(history)
block = f"{model_name}\n{description}\n"
block += df_history.to_csv(index=False).replace("\r\n", "\n").strip() + "\n\n"

with open(log_file, "a") as f:
    f.write(block)

print(f"Saved to {log_file}")

### Find mAP for classifying the correct overall family and add to log file

In [ ]:
def evaluate_families(loader):
    model.eval()
    all_labels, all_probs = [], []

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device, non_blocking=True)
            labels   = labels.to(device, non_blocking=True)
            logits   = model(features)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    families = list(top_level.values())  # 6 families
    n_clips  = len(all_labels)

    # Build family-level true labels and predicted scores
    true_family  = np.zeros((n_clips, len(families)))
    pred_family  = np.zeros((n_clips, len(families)))

    for i in range(n_clips):
        # True: which families are active for this clip
        true_indices = all_labels[i].nonzero()[0]
        for idx in true_indices:
            fam = get_top_level_family(vocab.iloc[idx]["mid"])
            if fam in families:
                true_family[i, families.index(fam)] = 1.0

        # Predicted: max prob per family
        for j, fam in enumerate(families):
            fam_indices = [k for k in range(200) if get_top_level_family(vocab.iloc[k]["mid"]) == fam]
            if fam_indices:
                pred_family[i, j] = all_probs[i, fam_indices].max()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UndefinedMetricWarning)
        mAP_overall = average_precision_score(true_family, pred_family, average="macro")
        mAP_per_fam = average_precision_score(true_family, pred_family, average=None)

    print(f"Overall family mAP: {mAP_overall:.3f}\n")
    for fam, score in zip(families, mAP_per_fam):
        print(f"  {fam:<30} {score:.3f}")
    
    return mAP_overall, dict(zip(families, mAP_per_fam)) 

In [ ]:
family_log_file = "family_mAP_log.csv"

# Run family evaluation on best model
model.load_state_dict(torch.load(f"{model_name}.pt"))
family_mAP_overall, family_mAP_per_fam = evaluate_families(val_loader)

# Save family mAP to log file
df_family = pd.DataFrame([{
    "family": fam,
    "mAP":    round(score, 4)
} for fam, score in family_mAP_per_fam.items()])
df_family.loc[len(df_family)] = {"family": "overall", "mAP": round(family_mAP_overall, 4)}

block = f"{model_name}\n{description}\n"
block += df_family.to_csv(index=False).replace("\r\n", "\n").strip() + "\n\n"

with open(family_log_file, "a") as f:
    f.write(block)

print(f"Saved to {family_log_file}")

Overall family mAP: 0.448

  Human sounds                   0.448
  Animal                         0.393
  Music                          0.743
  Natural sounds                 0.346
  Sounds of things               0.538
  Source-ambiguous sounds        0.219
Saved to family_mAP_log.csv


In [ ]:
from sklearn.metrics import roc_auc_score, label_ranking_average_precision_score
from scipy.stats import norm
import io

metrics_log_file = "/content/drive/MyDrive/ML_Final_Project/metrics_log.csv"

def evaluate_metrics(loader, model_name, description, log_file="metrics_log.csv"):
    model.eval()
    all_labels, all_probs = [], []

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device, non_blocking=True)
            labels   = labels.to(device, non_blocking=True)
            logits   = model(features)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.vstack(all_probs)   # (N, 200)
    all_labels = np.vstack(all_labels)  # (N, 200)

    # mAP
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UndefinedMetricWarning)
        mAP = average_precision_score(all_labels, all_probs, average="macro")

    # AUC per class → d-prime
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        auc_per_class = roc_auc_score(all_labels, all_probs, average=None)
    auc  = float(np.mean(auc_per_class))
    dprime = float(np.mean(norm.ppf(auc_per_class.clip(1e-7, 1 - 1e-7)) * np.sqrt(2)))

    # lwlrap — skip clips with no positive labels
    sample_weight = np.sum(all_labels > 0, axis=1)
    nonzero       = np.flatnonzero(sample_weight > 0)
    lwlrap = label_ranking_average_precision_score(
        all_labels[nonzero] > 0,
        all_probs[nonzero],
        sample_weight=sample_weight[nonzero]
    )

    print(f"mAP:    {mAP:.4f}")
    print(f"d':     {dprime:.4f}")
    print(f"lwlrap: {lwlrap:.4f}")

    # Save to log
    df = pd.DataFrame([{
        "metric": "mAP",    "value": round(float(mAP),    4)},
        {"metric": "d_prime", "value": round(float(dprime),  4)},
        {"metric": "lwlrap",  "value": round(float(lwlrap),  4)},
    ])
    block  = f"{model_name}\n{description}\n"
    block += df.to_csv(index=False).replace("\r\n", "\n").strip() + "\n\n"

    with open(log_file, "a") as f:
        f.write(block)

    print(f"Saved to {log_file}")
    return mAP, dprime, lwlrap

In [ ]:
model.load_state_dict(torch.load(f"{model_name}.pt"))
evaluate_metrics(val_loader, model_name, description, log_file=metrics_log_file)